# GReaT Synthetic Data Generation (Single Training, Multiple Generation)

This notebook generates 5 replicates of synthetic data using the GReaT library.

**Approach:**
- Train model ONCE (~2 hours)
- Generate 5 replicates with different seeds (~10-15 min each)
- Total time: ~2.5-3 hours (vs ~10 hours for full retraining)

**Variance captured:**
- Generation variance (sampling randomness)
- Note: Does not capture training variance (acceptable trade-off for efficiency)

**For n_samples=5000 experiments (default):**
- Upload `great_train_data_5k.csv` (3,500 training records)
- This matches n_train for CTGAN, DPBN, Synthpop, Independent Marginals

**IMPORTANT: Disk Space Management**
- Checkpointing is DISABLED (`save_steps=0`) to prevent disk overflow on Colab
- Cleanup functions run automatically between generations
- If you still run out of disk, restart runtime and re-run from training cell

Run this in Google Colab with GPU runtime (T4 recommended).

In [ ]:
# Install dependencies
!pip install be-great pandas -q
print("Dependencies installed")

In [ ]:
# Upload training data
from google.colab import files
uploaded = files.upload()
# Click "Choose Files" and select your training data CSV

In [ ]:
import pandas as pd
import numpy as np
import torch
import random
import gc
import shutil
import os
from be_great import GReaT
import warnings
warnings.filterwarnings('ignore')

def set_all_seeds(seed):
    """Set ALL random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print(f"  Seeds set to {seed}")

def cleanup_disk():
    """Clean up disk space - delete model checkpoints and cache."""
    deleted = []
    # Delete checkpoint directories and any great_* folders
    for d in os.listdir('.'):
        if d.startswith('great_') and os.path.isdir(d):
            shutil.rmtree(d)
            deleted.append(d)
    
    # Also clean up HuggingFace cache if needed
    hf_cache = os.path.expanduser('~/.cache/huggingface')
    if os.path.exists(hf_cache):
        cache_size = sum(os.path.getsize(os.path.join(dirpath, filename)) 
                        for dirpath, dirnames, filenames in os.walk(hf_cache) 
                        for filename in filenames) / (1024**3)
        if cache_size > 5:  # If cache > 5GB
            print(f"  HuggingFace cache is {cache_size:.1f}GB - consider clearing if disk is full")
    
    if deleted:
        print(f"  Deleted: {', '.join(deleted)}")
    
    # Force garbage collection
    gc.collect()
    
    # Clear CUDA cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"  CUDA cache cleared")

def check_disk_space():
    """Show available disk space."""
    import subprocess
    result = subprocess.run(['df', '-h', '.'], capture_output=True, text=True)
    lines = result.stdout.strip().split('\n')
    if len(lines) > 1:
        parts = lines[1].split()
        if len(parts) >= 4:
            avail = parts[3]
            print(f"  Disk: {avail} available")
            # Warn if low
            if avail.endswith('G'):
                gb = float(avail[:-1])
                if gb < 10:
                    print(f"  ⚠️  WARNING: Low disk space! Consider running cleanup_disk()")

# Clean up any leftover checkpoints from previous runs FIRST
print("Cleaning up any leftover files from previous runs...")
cleanup_disk()

# Check GPU
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected - training will be slow!")

check_disk_space()

# Load training data - MUST MATCH OTHER METHODS
# For n_samples=5000 experiments: use 'great_train_data_5k.csv' (3,500 rows)
# This ensures fair comparison with CTGAN, DPBN, Synthpop (all use n_train=3500)
TRAIN_FILE = 'great_train_data_5k.csv'  # <-- 3,500 training records (n_train for n_samples=5000)
df_train = pd.read_csv(TRAIN_FILE)
print(f"\nLoaded {len(df_train)} training records")
print(f"Columns: {list(df_train.columns)}")

In [ ]:
# Configuration
TRAINING_SEED = 42  # Fixed seed for training
GENERATION_SEEDS = [42, 123, 456, 789, 1011]  # Different seeds for each replicate
N_REPLICATES = 5
EPOCHS = 100
BATCH_SIZE = 32
LLM = 'distilgpt2'  # Smallest/fastest option

print("Configuration:")
print(f"  Training seed: {TRAINING_SEED}")
print(f"  Generation seeds: {GENERATION_SEEDS}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Model: {LLM}")
print(f"  N samples per replicate: {len(df_train)}")
print(f"\nApproach: Train ONCE, generate {N_REPLICATES} times")

In [ ]:
# STEP 1: Train model ONCE
print("="*60)
print("TRAINING MODEL (this takes ~2 hours)")
print("="*60)

check_disk_space()

# Set training seed
set_all_seeds(TRAINING_SEED)

# Create and train model - DISABLE CHECKPOINTING to save disk space
print("Creating GReaT model...")
model = GReaT(
    llm=LLM,
    experiment_dir='great_model',
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    seed=TRAINING_SEED,
    data_seed=TRAINING_SEED,
    save_steps=0,  # Disable checkpoint saving (prevents disk overflow on Colab)
)

print("Training... (this will take a while)")
print("NOTE: Checkpointing disabled to prevent disk overflow on Colab")
model.fit(df_train)
print("\nTraining complete!")
check_disk_space()

In [ ]:
# STEP 2: Generate 5 replicates with different seeds
generated_files = []

print("\n" + "="*60)
print("GENERATING 5 REPLICATES")
print("="*60)

for rep in range(1, N_REPLICATES + 1):
    seed = GENERATION_SEEDS[rep - 1]
    
    print(f"\n{'─'*40}")
    print(f"REPLICATE {rep}/{N_REPLICATES} (generation seed={seed})")
    print(f"{'─'*40}")
    
    # Check disk space before each generation
    check_disk_space()
    
    # Set generation seed
    set_all_seeds(seed)
    
    # Generate synthetic data
    print(f"  Generating {len(df_train)} synthetic records...")
    df_synth = model.sample(
        n_samples=len(df_train),
        temperature=0.7,
        k=100,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        drop_nan=True,
    )
    
    # Save
    filename = f'great_rep{rep}.csv'
    df_synth.to_csv(filename, index=False)
    generated_files.append(filename)
    print(f"  Saved to {filename} ({len(df_synth)} records)")
    
    # Quick sanity check - show sample values
    numeric_cols = df_synth.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        col = numeric_cols[0]
        print(f"  Sample {col} values: {df_synth[col].head(3).tolist()}")
    
    # Clear memory between generations
    del df_synth
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # Clean up any checkpoint files that may have been created
    cleanup_disk()

print(f"\n{'='*60}")
print("ALL REPLICATES COMPLETE!")
print(f"{'='*60}")
print(f"\nGenerated files: {generated_files}")

# Final cleanup
print("\nFinal cleanup...")
cleanup_disk()
check_disk_space()

In [ ]:
# Verify files are different
import hashlib

print("Verifying replicates are different:")
print("-" * 40)

hashes = []
for filename in generated_files:
    if os.path.exists(filename):
        with open(filename, 'rb') as f:
            h = hashlib.md5(f.read()).hexdigest()[:12]
        hashes.append(h)
        print(f"  {filename}: {h}")
    else:
        print(f"  {filename}: NOT FOUND")

unique_hashes = len(set(hashes))
if unique_hashes == len(generated_files):
    print(f"\n ALL {unique_hashes} FILES ARE DIFFERENT (seeding works!)")
else:
    print(f"\n WARNING: Only {unique_hashes} unique files out of {len(generated_files)}")

In [ ]:
# Download all files
from google.colab import files

for filename in generated_files:
    if os.path.exists(filename):
        files.download(filename)
        print(f"Downloaded {filename}")
    else:
        print(f"MISSING: {filename}")